# Submitting a RBFE Alchemiscale

In [1]:
import json

from alchemiscale import AlchemiscaleClient

In [2]:
asc = AlchemiscaleClient(
    "https://api.alchemiscale.org",
)
# Credentials are autodetected here

## Executing Our Network

In [3]:
BENCHMARK_SET = "jacs_set"
BENCHMARK_SYS_NAMES = ["thrombin", "tyk2"]

In [ ]:
%%script false --no-raise-error

# Now make your own scope and submit a network, 
# RUNNING MULTIPLE TIMES WILL CREATE MULTIPLE REPLICATES ON ALCHEMISCALE
keys = defaultdict()
for bench_sys in BENCHMARK_SYS_NAMES:
    if f'rbfe_pontibus_{BENCHMARK_SET}_{bench_sys}' not in keys:
        network = openfe.AlchemicalNetwork.from_json(
            file=f"outputs/alchemical_network_{BENCHMARK_SET}_{bench_sys}.json")
        scope = Scope('openff', 'ff14SB_openff_3_0_0_alpha0_opc3', f'rbfe_{BENCHMARK_SET}_{bench_sys}')
        keys[f'rbfe_pontibus_{BENCHMARK_SET}_{bench_sys}'] = asc.create_network(network, scope)

with open("alchemicalnetwork_scopekeys.json", "w") as f:
    json.dump({k: str(v) for k, v in keys.items()}, f, indent=2)

Output()

Output()

In [ ]:
%%script false --no-raise-error
n_repeats = 3

for name, an_sk in keys.items():
    tf_sks = asc.get_network_transformations(an_sk)
    tasks = asc.create_transformations_tasks(tf_sks * n_repeats)
    print(f"There are {len(tasks)} tasks")
    
    # Now we need to action the tasks before they can be picked up for compute
    asc.action_tasks(tasks, an_sk)

There are 84 tasks
There are 132 tasks


In [3]:
with open("alchemicalnetwork_scopekeys.json", "r") as f:
    keys = json.load(f)

for name, an_sk in keys.items():
    asc.get_network_status(an_sk)

AlchemicalNetwork-8f0230086292763f9be6b738348307e8-openff-ff14SB_openff_3_0_0_alpha0_opc3-rbfe_jacs_set_thrombin   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ status                                                           ┃                                        count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ complete                                                         │                                           84 │
│ running                                                          │                                            0 │
│ waiting                                                          │                                            0 │
│ error                                                            │                                            0 │
│ invalid                                                          │                                            0 │
│ deleted                                                          │                                            0 │
└──────────────────────────────────────────────────────────────────┴──────────────────────────────────────────────┘

AlchemicalNetwork-a1efafd1298d735c7032cebe52be9665-openff-ff14SB_openff_3_0_0_alpha0_opc3-rbfe_jacs_set_tyk2       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ status                                                           ┃                                        count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ complete                                                         │                                          132 │
│ running                                                          │                                            0 │
│ waiting                                                          │                                            0 │
│ error                                                            │                                            0 │
│ invalid                                                          │                                            0 │
│ deleted                                                          │                                            0 │
└──────────────────────────────────────────────────────────────────┴──────────────────────────────────────────────┘

## Checking Results

In [1]:
with open("alchemicalnetwork_scopekeys.json", "r") as f:
    keys = json.load(f)

for name, an_sk in keys.items():
    asc.get_network_status(an_sk)

NameError: name 'json' is not defined

In [ ]:
failed_tasks = asc.get_network_tasks(an_sk, status="error")
failures = asc.get_task_failures(failed_tasks[0])
failures

In [ ]:
print(failures[0].protocol_unit_failures[0].traceback)

In [ ]:
# asc.set_tasks_status(failed_tasks[:1], 'waiting')

# Add Restart Pattern

In [ ]:
with open("alchemicalnetwork_scopekeys.json", "r") as f:
    keys = json.load(f)

for name, an_sk in keys.items():
    print(name)
#    asc.add_task_restart_patterns(an_sk, [r"No compatible CUDA device is available"], 5)
#    asc.remove_task_restart_patterns(an_sk, ["MemoryError: Unable to allocate \d+ GiB"])
#    asc.clear_task_restart_patterns(an_sk)

rbfe_pontibus_jacs_set_thrombin
rbfe_pontibus_jacs_set_tyk2


## Increase Priority

In [8]:
with open("alchemicalnetwork_scopekeys.json", "r") as f:
    keys = json.load(f)

for name, an_sk in keys.items():
    print(name)
    waiting_tasks = asc.get_network_tasks(an_sk, status="waiting")
#    asc.set_tasks_priority(waiting_tasks, 10)
#    asc.get_tasks_priority(an_sk)

rbfe_pontibus_jacs_set_thrombin


/Users/jenniferclark/mamba/envs/openfe-benchmarks-test/lib/python3.13/site-packages/httpx/_models.py:408: DeprecationWarning: Use 'content=<...>' to upload raw bytes/text content.
  headers, stream = encode_request(


rbfe_pontibus_jacs_set_tyk2


/Users/jenniferclark/mamba/envs/openfe-benchmarks-test/lib/python3.13/site-packages/httpx/_models.py:408: DeprecationWarning: Use 'content=<...>' to upload raw bytes/text content.
  headers, stream = encode_request(
